In [1]:
!nvidia-smi

Thu Nov 12 08:16:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             44W /   70W |    9029MiB /  15360MiB |     63%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DIR = Path(
    '/content/drive/MyDrive/CaReFlow/checkpoints'
)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

print("Google Drive 保存目录：", DRIVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive 保存目录： /content/drive/MyDrive/CaReFlow/checkpoints


In [3]:
!git clone https://github.com/Scream9371/CaReFlow.git

Cloning into 'CaReFlow'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 37 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 27.48 KiB | 461.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.


In [4]:
%cd /content/CaReFlow/datasets/
!bash ./download_datasets.sh
# 检查下载结果
!ls -lh ./{mosi,mosei}.pkl
!wc -c ./{mosi,mosei}.pkl

/content/CaReFlow/datasets
Downloading...
From: https://drive.google.com/uc?id=12HbavGOtoVCqicvSYWl3zImli5Jz0Nou
To: /content/CaReFlow/datasets/mosi.pkl
100% 14.2M/14.2M [00:00<00:00, 41.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1VJhSc2TGrPU8zJSVTYwn5kfuG47VaNQ3
From (redirected): https://drive.google.com/uc?id=1VJhSc2TGrPU8zJSVTYwn5kfuG47VaNQ3&confirm=t&uuid=79e9e530-7a72-47fa-afd7-a8ef4d98a77c
To: /content/CaReFlow/datasets/mosei.pkl
100% 508M/508M [00:10<00:00, 50.5MB/s] 
-rw-r--r-- 1 root root 485M Nov 11  2020 ./mosei.pkl
-rw-r--r-- 1 root root  14M Nov 11  2020 ./mosi.pkl
 14158784 ./mosi.pkl
507823345 ./mosei.pkl
521982129 total


In [5]:
# 1. 将 DeBERTa-v3-base 权重和配置文件下载到正确的 microsoft/deberta-v3-base 路径下
import os
from huggingface_hub import snapshot_download

target_model_dir = "/content/CaReFlow/microsoft/deberta-v3-base"
os.makedirs(target_model_dir, exist_ok=True)

print("正在下载 DeBERTa-v3-base 模型至官方匹配路径...")
snapshot_download(
    repo_id="microsoft/deberta-v3-base",
    local_dir=target_model_dir,
    ignore_patterns=["*.msgpack", "*.h5", "*.ot"]
)

print("下载并保存完毕，文件列表如下：")
!ls -lh {target_model_dir}

正在下载 DeBERTa-v3-base 模型至官方匹配路径...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

下载并保存完毕，文件列表如下：
total 357M
-rw-r--r-- 1 root root    0 Nov 12 08:17 1.txt
-rw-r--r-- 1 root root  579 Nov 12 08:17 config.json
-rw-r--r-- 1 root root 354M Nov 12 08:17 pytorch_model.bin
-rw-r--r-- 1 root root 3.4K Nov 12 08:17 README.md
-rw-r--r-- 1 root root 2.4M Nov 12 08:17 spm.model
-rw-r--r-- 1 root root   52 Nov 12 08:17 tokenizer_config.json


In [6]:
!ls -lh /content/CaReFlow/microsoft/deberta-v3-base

total 357M
-rw-r--r-- 1 root root    0 Nov 12 08:17 1.txt
-rw-r--r-- 1 root root  579 Nov 12 08:17 config.json
-rw-r--r-- 1 root root 354M Nov 12 08:17 pytorch_model.bin
-rw-r--r-- 1 root root 3.4K Nov 12 08:17 README.md
-rw-r--r-- 1 root root 2.4M Nov 12 08:17 spm.model
-rw-r--r-- 1 root root   52 Nov 12 08:17 tokenizer_config.json


In [7]:
!python -m pip install "transformers==4.57.6"
!python -c "import torch, transformers; print('PyTorch:', torch.__version__); print('Transformers:', transformers.__version__); print('CUDA:', torch.cuda.is_available())"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 81.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.9 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.00/3 [huggingface-hub]
  Attempting uninstall: tokenizers━━━━━━━━━━━━━━ 0/3 [huggingface-hub]
    Found existing installation: tokenizers 0.23.30/3 [huggingface-hub]
    Uninstalling tokenizers-0.23.3:━━━━━━━━━ 0/3 [huggingface-hub]
      Successfully uninstalled tokenizers-0.23.3 0/3 [huggingface-hub]
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [tokenizers]
    Found existing installation: transformers 5.22.1━━━━━━━━━━ 1/3 [tokenizers]
    Uninstalling transformers-5.22.1:m╸━━━━━━━━━━━━━ 2/3 [transformers]
      Successfully uninstalled transformers-5.2

In [8]:
%%bash
set -e
cd /content/CaReFlow
python -u train_reflow_new.py \
    --dataset mosi \
    --n_epochs 100 \
    --train_batch_size 32 \
    --learning_rate 10e-6 \
    --ratio 4 \
    --dropout_prob 0.5 \
    --inter_dim 150 \
    --share_dim 100 \
    --transformer_layer 3 \
    --step_size 2 \
    --loss_b_ratio 0.1 \
    --loss_f_ratio 0.2 \
    --eps 1e-3 \
    --save_model \
    --output_dir /content/CaReFlow/checkpoints
cp /content/CaReFlow/checkpoints/careflow_mosi_best.pt /content/drive/MyDrive/CaReFlow/checkpoints/


Seed: 128


I0000 00:00:1794471495.446788   10744 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794471499.103654   10744 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Traceback (most recent call last):
  File "/content/CaReFlow/train_reflow_new.py", line 397, in <module>
    main()
    ~~~~^^
  File "/content/CaReFlow/train_reflow_new.py", line 391, in main
    model, optimizer, scheduler = prep_for_training(num_train_optimization_steps)
                                  ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/CaReFlow/train

CalledProcessError: Command 'b'set -e\ncd /content/CaReFlow\npython -u train_reflow_new.py \\\n    --dataset mosi \\\n    --n_epochs 100 \\\n    --train_batch_size 32 \\\n    --learning_rate 10e-6 \\\n    --ratio 4 \\\n    --dropout_prob 0.5 \\\n    --inter_dim 150 \\\n    --share_dim 100 \\\n    --transformer_layer 3 \\\n    --step_size 2 \\\n    --loss_b_ratio 0.1 \\\n    --loss_f_ratio 0.2 \\\n    --eps 1e-3 \\\n    --save_model \\\n    --output_dir /content/CaReFlow/checkpoints\ncp /content/CaReFlow/checkpoints/careflow_mosi_best.pt /content/drive/MyDrive/CaReFlow/checkpoints/\n'' returned non-zero exit status 1.

In [ ]:
%%bash
set -e
cd /content/CaReFlow
python -u train_reflow_new.py \
    --dataset mosei \
    --n_epochs 100 \
    --train_batch_size 32 \
    --learning_rate 10e-6 \
    --ratio 4 \
    --dropout_prob 0.5 \
    --inter_dim 150 \
    --share_dim 100 \
    --transformer_layer 3 \
    --step_size 2 \
    --loss_b_ratio 0.1 \
    --loss_f_ratio 0.2 \
    --eps 1e-3 \
    --save_model \
    --output_dir /content/CaReFlow/checkpoints
cp /content/CaReFlow/checkpoints/careflow_mosei_best.pt /content/drive/MyDrive/CaReFlow/checkpoints/
